# Avaliação de Robustez — all-MiniLM-L12-v2

Este notebook avalia a **robustez** do modelo `all-MiniLM-L12-v2` em cenários adversos e de variação linguística,
simulando condições reais de uso em sistemas de Recuperação Aumentada por Geração (RAG) para a **EJA**.

---

## Estrutura do Notebook

| Seção | Teste | Objetivo |
|-------|-------|----------|
| 2 | Erros de Digitação e Ortografia | Verificar sensibilidade a acentuação e erros comuns |
| 3 | Negação | ⚠️ Identificar risco crítico em RAG |
| 4 | Paráfrases | Verificar invariância semântica |
| 5 | Textos Longos / Truncamento | Avaliar impacto do comprimento |
| 6 | Mudança de Domínio | Avaliar generalização fora do domínio EJA |
| 7 | Sumário de Robustez | Consolidação de métricas e riscos |

---
## Seção 1 — Configuração e Carregamento do Modelo

In [ ]:
# =============================================================
# Importações e configuração global
# =============================================================
import numpy as np
import pandas as pd
import json
import os
import warnings
import itertools

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings('ignore')

# Estilo visual consistente
sns.set_theme(style='whitegrid', palette='muted')
matplotlib.rcParams['figure.dpi'] = 120
matplotlib.rcParams['font.size'] = 11

# Diretórios de saída (caminhos relativos ao diretório notebooks/)
PLOTS_DIR   = '../results/plots'
METRICS_DIR = '../results/metrics'
os.makedirs(PLOTS_DIR,   exist_ok=True)
os.makedirs(METRICS_DIR, exist_ok=True)

print('✅ Bibliotecas importadas com sucesso.')

In [ ]:
# =============================================================
# Carregamento do modelo all-MiniLM-L12-v2
# =============================================================
MODEL_NAME = 'sentence-transformers/all-MiniLM-L12-v2'

try:
    model = SentenceTransformer(MODEL_NAME)
    print(f'✅ Modelo carregado: {MODEL_NAME}')
    print(f'   Dimensão dos embeddings : {model.get_sentence_embedding_dimension()}')
    print(f'   Comprimento máximo (tokens): {model.max_seq_length}')
except Exception as e:
    print(f'❌ Erro ao carregar o modelo: {e}')
    raise

# Função utilitária: codifica texto(s) e retorna embeddings
def encode(texts):
    """Codifica uma lista de textos e retorna array de embeddings numpy."""
    if isinstance(texts, str):
        texts = [texts]
    return model.encode(texts, convert_to_numpy=True, show_progress_bar=False)

# Função utilitária: similaridade cosseno entre dois vetores
def cos_sim(a, b):
    """Calcula a similaridade cosseno entre dois vetores 1-D."""
    return float(cosine_similarity(a.reshape(1, -1), b.reshape(1, -1))[0, 0])

print('✅ Funções utilitárias definidas.')

---
## Seção 2 — Teste de Erros de Digitação e Ortografia

Avaliamos se o modelo mantém similaridades altas entre sentenças com:
- **sem_acento**: acentos removidos (ção→cao, ã→a, é→e, etc.)
- **minuscula**: tudo em minúsculo e sem acentos
- **erro_tipico**: erros de digitação comuns no contexto EJA

> **Esperado:** modelo robusto deve manter similaridade ≥ 0.90 mesmo com variações ortográficas.

In [ ]:
# =============================================================
# Dados: 10 sentenças base com 3 variantes ortográficas cada
# =============================================================
orthography_tests = [
    {
        "original"   : "Como funciona a Educação de Jovens e Adultos?",
        "sem_acento" : "Como funciona a Educacao de Jovens e Adultos?",
        "minuscula"  : "como funciona a educacao de jovens e adultos?",
        "erro_tipico": "Como funciona a Educaçao de Jovens e Adultos?"
    },
    {
        "original"   : "Quais documentos são necessários para a matrícula na EJA?",
        "sem_acento" : "Quais documentos sao necessarios para a matricula na EJA?",
        "minuscula"  : "quais documentos sao necessarios para a matricula na eja?",
        "erro_tipico": "Quais documentos são nescessarios pra a matricula na EJA?"
    },
    {
        "original"   : "O aluno precisa ter mais de 15 anos para se matricular.",
        "sem_acento" : "O aluno precisa ter mais de 15 anos para se matricular.",
        "minuscula"  : "o aluno precisa ter mais de 15 anos para se matricular.",
        "erro_tipico": "O aluno preciza ter mais de 15 anos pra se matricular."
    },
    {
        "original"   : "A EJA oferece certificação equivalente ao ensino regular.",
        "sem_acento" : "A EJA oferece certificacao equivalente ao ensino regular.",
        "minuscula"  : "a eja oferece certificacao equivalente ao ensino regular.",
        "erro_tipico": "A EJA oferece sertificação equivalente ao ensino regular."
    },
    {
        "original"   : "As aulas da EJA são gratuitas nas escolas públicas.",
        "sem_acento" : "As aulas da EJA sao gratuitas nas escolas publicas.",
        "minuscula"  : "as aulas da eja sao gratuitas nas escolas publicas.",
        "erro_tipico": "As aulas da EJA são gratuitas nas escola publicas."
    },
    {
        "original"   : "Como obter o histórico escolar da educação de adultos?",
        "sem_acento" : "Como obter o historico escolar da educacao de adultos?",
        "minuscula"  : "como obter o historico escolar da educacao de adultos?",
        "erro_tipico": "Como obter o historico ecolar da educação de adultos?"
    },
    {
        "original"   : "Posso transferir minha matrícula de uma escola para outra?",
        "sem_acento" : "Posso transferir minha matricula de uma escola para outra?",
        "minuscula"  : "posso transferir minha matricula de uma escola para outra?",
        "erro_tipico": "Posso transeferir minha matricula de uma escola pra outra?"
    },
    {
        "original"   : "Qual é o horário das aulas na modalidade EJA noturna?",
        "sem_acento" : "Qual e o horario das aulas na modalidade EJA noturna?",
        "minuscula"  : "qual e o horario das aulas na modalidade eja noturna?",
        "erro_tipico": "Qual é o orario das aulas na modalidade EJA noturna?"
    },
    {
        "original"   : "A frequência mínima exigida na EJA é de 75 por cento.",
        "sem_acento" : "A frequencia minima exigida na EJA e de 75 por cento.",
        "minuscula"  : "a frequencia minima exigida na eja e de 75 por cento.",
        "erro_tipico": "A frequencia minima exijida na EJA e de 75 porcento."
    },
    {
        "original"   : "O exame do ENCCEJA substitui o diploma da EJA?",
        "sem_acento" : "O exame do ENCCEJA substitui o diploma da EJA?",
        "minuscula"  : "o exame do encceja substitui o diploma da eja?",
        "erro_tipico": "O exame do ENCEJA substitue o diploma da EJA?"
    },
]

print(f'✅ {len(orthography_tests)} grupos de testes ortográficos definidos.')

In [ ]:
# =============================================================
# Cálculo de similaridades ortográficas
# =============================================================
variant_keys  = ['sem_acento', 'minuscula', 'erro_tipico']
variant_labels = {
    'sem_acento' : 'Sem Acento',
    'minuscula'  : 'Minúscula S/ Acento',
    'erro_tipico': 'Erro Típico'
}

ortho_records = []  # linha por grupo e variante

for i, group in enumerate(orthography_tests):
    orig_emb = encode(group['original'])[0]
    for vk in variant_keys:
        var_emb = encode(group[vk])[0]
        sim = cos_sim(orig_emb, var_emb)
        ortho_records.append({
            'grupo'           : i + 1,
            'original'        : group['original'][:45] + '…',
            'variante_tipo'   : vk,
            'variante_label'  : variant_labels[vk],
            'texto_variante'  : group[vk],
            'similaridade'    : round(sim, 4)
        })

df_ortho = pd.DataFrame(ortho_records)

# Resumo por tipo de variante
resumo_ortho = df_ortho.groupby('variante_label')['similaridade'].agg(
    media='mean', minimo='min', maximo='max', std='std'
).round(4)

print('📊 Resumo de Similaridade por Tipo de Variante Ortográfica')
print('=' * 60)
print(resumo_ortho.to_string())

# Variante mais problemática por grupo
print('\n⚠️  Variante com MENOR similaridade por grupo:')
for g in range(1, 11):
    sub = df_ortho[df_ortho['grupo'] == g]
    row = sub.loc[sub['similaridade'].idxmin()]
    print(f'  Grupo {g:2d}: [{row["variante_label"]}] sim={row["similaridade"]:.4f}')

In [ ]:
# =============================================================
# Gráfico: barras agrupadas — similaridade por variante ortográfica
# =============================================================
fig, ax = plt.subplots(figsize=(14, 6))

pivot = df_ortho.pivot(index='grupo', columns='variante_label', values='similaridade')
pivot.plot(kind='bar', ax=ax, edgecolor='white', width=0.75,
           color=['#4c78a8', '#f58518', '#e45756'])

ax.axhline(0.90, color='green', linestyle='--', linewidth=1.5,
           label='Limiar de robustez (0.90)')
ax.axhline(0.80, color='red', linestyle=':', linewidth=1.5,
           label='Alerta (0.80)')

ax.set_title('Robustez Ortográfica — Similaridade com a Sentença Original\n'
             'por Tipo de Variante e Grupo', fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Grupo de Sentença', fontsize=11)
ax.set_ylabel('Similaridade Cosseno', fontsize=11)
ax.set_xticklabels([f'G{i}' for i in range(1, 11)], rotation=0)
ax.set_ylim(0.6, 1.02)
ax.legend(loc='lower right', fontsize=9)
ax.grid(axis='y', alpha=0.4)

plt.tight_layout()
ortho_path = f'{PLOTS_DIR}/05_orthography_robustness.png'
plt.savefig(ortho_path, bbox_inches='tight')
plt.show()
print(f'✅ Gráfico salvo em: {ortho_path}')

---
## Seção 3 — Teste de Negação ⚠️ RISCO CRÍTICO

Pares onde a presença/ausência de **"não"** inverte completamente o significado.

**Por que é crítico para RAG?**  
Se o modelo atribui **alta similaridade** entre uma afirmação e sua negação, o sistema de recuperação pode retornar documentos com informação *oposta* à pergunta do usuário — um risco grave em contextos educacionais e legais.

- **Similaridade > 0.90**: zona de perigo máximo  
- **Similaridade > 0.85**: preocupante para RAG  
- **Similaridade < 0.70**: modelo distingue bem a negação

In [ ]:
# =============================================================
# Dados: 20 pares de sentenças — afirmação vs. negação
# =============================================================
negation_pairs = [
    # (afirmação, negação, tópico)
    ("O estudante pode realizar a matrícula.",
     "O estudante não pode realizar a matrícula.",
     "matrícula"),
    ("A escola oferece ensino gratuito.",
     "A escola não oferece ensino gratuito.",
     "gratuidade"),
    ("O aluno concluiu o ensino médio.",
     "O aluno não concluiu o ensino médio.",
     "conclusão"),
    ("A EJA aceita transferência de outra escola.",
     "A EJA não aceita transferência de outra escola.",
     "transferência"),
    ("Preciso de documentos para me matricular.",
     "Não preciso de documentos para me matricular.",
     "documentos"),
    ("O aluno foi aprovado na avaliação final.",
     "O aluno não foi aprovado na avaliação final.",
     "aprovação"),
    ("A EJA oferece certificado de conclusão válido.",
     "A EJA não oferece certificado de conclusão válido.",
     "certificação"),
    ("Posso fazer a matrícula online.",
     "Não posso fazer a matrícula online.",
     "matrícula_digital"),
    ("O curso da EJA tem duração de dois anos.",
     "O curso da EJA não tem duração de dois anos.",
     "duração"),
    ("É necessário ter 18 anos para ingressar na EJA.",
     "Não é necessário ter 18 anos para ingressar na EJA.",
     "idade_mínima"),
    ("A frequência é obrigatória nas aulas da EJA.",
     "A frequência não é obrigatória nas aulas da EJA.",
     "frequência"),
    ("O aluno pode trancar a matrícula.",
     "O aluno não pode trancar a matrícula.",
     "trancamento"),
    ("A EJA funciona no período noturno.",
     "A EJA não funciona no período noturno.",
     "período"),
    ("O ENCCEJA substitui o diploma da EJA.",
     "O ENCCEJA não substitui o diploma da EJA.",
     "ENCCEJA"),
    ("É possível estudar na EJA e trabalhar ao mesmo tempo.",
     "Não é possível estudar na EJA e trabalhar ao mesmo tempo.",
     "conciliação"),
    ("A escola possui sala de recursos para alunos com deficiência.",
     "A escola não possui sala de recursos para alunos com deficiência.",
     "acessibilidade"),
    ("O aluno recebe material didático gratuitamente.",
     "O aluno não recebe material didático gratuitamente.",
     "material_didático"),
    ("A EJA aceita alunos com histórico de reprovação.",
     "A EJA não aceita alunos com histórico de reprovação.",
     "reprovação"),
    ("O aluno pode solicitar reingresso após abandono.",
     "O aluno não pode solicitar reingresso após abandono.",
     "reingresso"),
    ("A secretaria escolar emite declaração de matrícula.",
     "A secretaria escolar não emite declaração de matrícula.",
     "declaração"),
]

print(f'✅ {len(negation_pairs)} pares de negação definidos.')

In [ ]:
# =============================================================
# Cálculo de similaridades dos pares de negação
# =============================================================
negation_records = []

for idx, (sent_a, sent_b, topic) in enumerate(negation_pairs):
    emb_a = encode(sent_a)[0]
    emb_b = encode(sent_b)[0]
    sim = cos_sim(emb_a, emb_b)
    negation_records.append({
        'par'         : idx + 1,
        'sentence_a'  : sent_a,
        'sentence_b'  : sent_b,
        'topico'      : topic,
        'similaridade': round(sim, 4)
    })

df_neg = pd.DataFrame(negation_records)

# Classificação de risco
def risco_negacao(sim):
    if sim >= 0.90: return '🔴 ALTO'
    if sim >= 0.85: return '🟠 MÉDIO'
    if sim >= 0.70: return '🟡 BAIXO'
    return '🟢 ADEQUADO'

df_neg['risco'] = df_neg['similaridade'].apply(risco_negacao)

print('📊 Resultados dos Pares de Negação')
print('=' * 80)
for _, row in df_neg.iterrows():
    print(f'  Par {row["par"]:2d} [{row["topico"]:20s}] sim={row["similaridade"]:.4f}  {row["risco"]}')

print(f'\n📈 Similaridade média global: {df_neg["similaridade"].mean():.4f}')
print(f'   Pares com sim > 0.85 (problemáticos para RAG): '
      f'{(df_neg["similaridade"] > 0.85).sum()} de {len(df_neg)}')

In [ ]:
# =============================================================
# Gráfico: barras horizontais — similaridade por par de negação
# =============================================================
fig, ax = plt.subplots(figsize=(12, 10))

# Paleta de cores baseada no risco
cores = []
for sim in df_neg['similaridade']:
    if sim >= 0.90:  cores.append('#d62728')  # vermelho — alto risco
    elif sim >= 0.85: cores.append('#ff7f0e')  # laranja — médio risco
    elif sim >= 0.70: cores.append('#ffdd57')  # amarelo — baixo risco
    else:             cores.append('#2ca02c')  # verde — adequado

y_pos = np.arange(len(df_neg))
labels = [f"P{row['par']:02d}: {row['topico']}" for _, row in df_neg.iterrows()]

bars = ax.barh(y_pos, df_neg['similaridade'], color=cores, edgecolor='white', height=0.7)

# Linha de perigo
ax.axvline(0.90, color='darkred',  linestyle='--', linewidth=2,
           label='Zona de Perigo (0.90)')
ax.axvline(0.85, color='darkorange', linestyle=':', linewidth=1.5,
           label='Preocupante para RAG (0.85)')

# Rótulos de valor
for bar, sim in zip(bars, df_neg['similaridade']):
    ax.text(sim + 0.003, bar.get_y() + bar.get_height() / 2,
            f'{sim:.3f}', va='center', ha='left', fontsize=8.5)

ax.set_yticks(y_pos)
ax.set_yticklabels(labels, fontsize=9)
ax.set_xlim(0.5, 1.05)
ax.set_xlabel('Similaridade Cosseno (Original ↔ Negação)', fontsize=11)
ax.set_title('⚠️  Teste de Negação — Similaridade entre Afirmação e Sua Negação\n'
             'Alta similaridade = risco para sistemas RAG', fontsize=12,
             fontweight='bold', pad=12)

# Legenda manual para cores de risco
patches = [
    mpatches.Patch(color='#d62728', label='Alto Risco (≥ 0.90)'),
    mpatches.Patch(color='#ff7f0e', label='Médio Risco (≥ 0.85)'),
    mpatches.Patch(color='#ffdd57', label='Baixo Risco (≥ 0.70)'),
    mpatches.Patch(color='#2ca02c', label='Adequado (< 0.70)'),
]
legend1 = ax.legend(handles=patches, loc='lower right', fontsize=8.5,
                    title='Nível de Risco', title_fontsize=9)
ax.add_artist(legend1)
ax.legend(loc='upper left', fontsize=9)

ax.grid(axis='x', alpha=0.3)
plt.tight_layout()

neg_path = f'{PLOTS_DIR}/05_negation_similarity.png'
plt.savefig(neg_path, bbox_inches='tight')
plt.show()
print(f'✅ Gráfico salvo em: {neg_path}')

In [ ]:
# =============================================================
# Pares mais problemáticos para RAG (similaridade > 0.85)
# =============================================================
df_problematic = df_neg[df_neg['similaridade'] > 0.85].sort_values(
    'similaridade', ascending=False
)

print(f'🚨 Pares com similaridade > 0.85 — MAIS PROBLEMÁTICOS PARA RAG')
print('=' * 80)
for _, row in df_problematic.iterrows():
    print(f'\n  Tópico    : {row["topico"]}')
    print(f'  Sentença A: {row["sentence_a"]}')
    print(f'  Sentença B: {row["sentence_b"]}')
    print(f'  Similaridade: {row["similaridade"]:.4f}  {row["risco"]}')
    print(f'  Risco RAG: Documento com informação OPOSTA pode ser recuperado!')

if df_problematic.empty:
    print('  ✅ Nenhum par com similaridade > 0.85 encontrado.')

---
## Seção 4 — Teste de Paráfrases

Avaliamos se o modelo **agrupa corretamente** sentenças semanticamente equivalentes,
mesmo com vocabulário e estrutura sintática diferentes.

Para cada grupo de 4 paráfrases calculamos:
- Similaridade par-a-par intra-grupo
- Consistência de recuperação: as 4 variantes recuperam os mesmos documentos do corpus?

In [ ]:
# =============================================================
# Dados: 8 grupos de paráfrases sobre EJA
# =============================================================
paraphrase_groups = [
    {
        "topic": "matricula_eja",
        "sentences": [
            "Como faço para entrar na EJA?",
            "Como posso me matricular na Educação de Jovens e Adultos?",
            "Quais são os procedimentos para estudar pela EJA?",
            "Quero voltar a estudar. Como faço?"
        ]
    },
    {
        "topic": "documentos_necessarios",
        "sentences": [
            "Que documentos preciso para me matricular na EJA?",
            "Quais papéis devo levar para fazer a matrícula?",
            "Documentação necessária para ingressar na EJA?",
            "O que trazer para realizar a matrícula na educação de adultos?"
        ]
    },
    {
        "topic": "certificado_conclusao",
        "sentences": [
            "A EJA fornece certificado válido ao terminar?",
            "Vou receber diploma ao concluir a EJA?",
            "O certificado da EJA tem validade legal?",
            "Qual documento recebo quando termino a EJA?"
        ]
    },
    {
        "topic": "idade_minima",
        "sentences": [
            "Qual a idade mínima para estudar na EJA?",
            "A partir de que idade posso me matricular na EJA?",
            "Tenho 15 anos, posso entrar na EJA?",
            "Existe limite de idade para ingressar na educação de jovens e adultos?"
        ]
    },
    {
        "topic": "horario_aulas",
        "sentences": [
            "Que horas são as aulas da EJA?",
            "A EJA funciona de noite?",
            "Qual o turno das aulas na educação de adultos?",
            "Em que período do dia ocorrem as aulas da EJA?"
        ]
    },
    {
        "topic": "gratuidade",
        "sentences": [
            "A EJA é paga ou gratuita?",
            "Preciso pagar alguma coisa para estudar na EJA?",
            "O ensino na EJA tem algum custo?",
            "A educação de jovens e adultos é de graça?"
        ]
    },
    {
        "topic": "transferencia",
        "sentences": [
            "Posso transferir minha matrícula para outra escola de EJA?",
            "Como faço para mudar de escola dentro da EJA?",
            "É possível continuar na EJA em outra unidade escolar?",
            "Posso pedir transferência entre escolas que oferecem EJA?"
        ]
    },
    {
        "topic": "reprovacao_frequencia",
        "sentences": [
            "Qual a frequência mínima exigida na EJA?",
            "Quantas faltas posso ter sem ser reprovado na EJA?",
            "A EJA exige presença mínima nas aulas?",
            "Serei reprovado por falta na EJA?"
        ]
    },
]

print(f'✅ {len(paraphrase_groups)} grupos de paráfrases definidos.')

In [ ]:
# =============================================================
# Corpus de recuperação para teste de consistência (10 documentos)
# =============================================================
eja_corpus = [
    "Para se matricular na EJA, o interessado deve comparecer à secretaria escolar com RG, CPF e comprovante de residência. A matrícula pode ser feita ao longo do ano.",
    "A Educação de Jovens e Adultos é totalmente gratuita nas escolas públicas municipais e estaduais. Não há cobrança de taxas, mensalidades ou materiais.",
    "O certificado de conclusão da EJA tem validade legal em todo o território nacional, equivalendo ao diploma do ensino fundamental ou médio regular.",
    "Para ingressar na EJA do ensino fundamental, o estudante deve ter no mínimo 15 anos. Para o ensino médio, a idade mínima é de 18 anos.",
    "As aulas da EJA são oferecidas principalmente no período noturno, das 19h às 22h, para atender trabalhadores e pessoas com outras obrigações diurnas.",
    "A transferência entre escolas de EJA é permitida. O aluno deve solicitar na secretaria atual o histórico escolar e a declaração de transferência.",
    "A frequência mínima exigida na EJA é de 75% das aulas. Faltas acima desse limite podem resultar em reprovação por infrequência.",
    "O ENCCEJA (Exame Nacional para Certificação de Competências de Jovens e Adultos) é um exame alternativo à EJA que permite obter certificação sem cursar as aulas.",
    "Alunos da EJA que precisam trancar a matrícula devem comunicar a escola. O reingresso é facilitado com a documentação do período anterior.",
    "A EJA aceita estudantes com histórico de reprovação no ensino regular. O ingresso é baseado na idade e não no histórico escolar anterior.",
]

corpus_embs = encode(eja_corpus)
print(f'✅ Corpus codificado: {corpus_embs.shape[0]} documentos × {corpus_embs.shape[1]} dimensões')

In [ ]:
# =============================================================
# Cálculo de similaridades par-a-par intra-grupo e consistência
# =============================================================
para_summary = []  # resumo por grupo
para_matrices = {} # matrizes de similaridade por grupo

TOP_K = 3  # top-3 documentos a comparar

for group in paraphrase_groups:
    topic     = group['topic']
    sentences = group['sentences']
    embs      = encode(sentences)

    # Matriz de similaridade intra-grupo
    sim_matrix = cosine_similarity(embs, embs)
    para_matrices[topic] = sim_matrix

    # Similaridade média (excluindo diagonal)
    n = len(sentences)
    mask = ~np.eye(n, dtype=bool)
    mean_sim = sim_matrix[mask].mean()
    min_sim  = sim_matrix[mask].min()

    # Consistência de recuperação: top-K de cada sentença
    sims_corpus = cosine_similarity(embs, corpus_embs)
    top3_sets = [set(np.argsort(sims_corpus[i])[::-1][:TOP_K]) for i in range(n)]
    # Interseção de todos os top-3
    intersection = top3_sets[0].intersection(*top3_sets[1:])
    consistency  = len(intersection) / TOP_K  # proporção de docs comuns

    para_summary.append({
        'topic'            : topic,
        'mean_intra_sim'   : round(float(mean_sim), 4),
        'min_intra_sim'    : round(float(min_sim), 4),
        'retrieval_consistency': round(consistency, 4),
        'docs_comuns_top3' : len(intersection)
    })

df_para = pd.DataFrame(para_summary)

print('📊 Resultados de Paráfrases')
print('=' * 70)
print(df_para.to_string(index=False))
print(f'\n📈 Média global (similaridade intra-grupo): {df_para["mean_intra_sim"].mean():.4f}')
print(f'   Consistência de recuperação média      : {df_para["retrieval_consistency"].mean():.4f}')

In [ ]:
# =============================================================
# Gráfico: heatmaps de similaridade par-a-par por grupo
# =============================================================
n_groups = len(paraphrase_groups)
ncols = 4
nrows = 2

fig, axes = plt.subplots(nrows, ncols, figsize=(18, 9))
axes = axes.flatten()

short_labels = ['S1', 'S2', 'S3', 'S4']  # rótulos compactos

for idx, group in enumerate(paraphrase_groups):
    topic  = group['topic']
    matrix = para_matrices[topic]
    ax     = axes[idx]

    sns.heatmap(
        matrix,
        ax=ax,
        vmin=0.5, vmax=1.0,
        cmap='YlOrRd',
        annot=True,
        fmt='.2f',
        linewidths=0.5,
        xticklabels=short_labels,
        yticklabels=short_labels,
        cbar=(idx == ncols - 1 or idx == n_groups - 1)
    )

    mean_sim = df_para.loc[df_para['topic'] == topic, 'mean_intra_sim'].values[0]
    ax.set_title(f'{topic}\n(μ={mean_sim:.3f})', fontsize=9, fontweight='bold')

# Ocultar eixo extra se n_groups < nrows*ncols
for j in range(n_groups, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Teste de Paráfrases — Similaridade Par-a-Par Intra-Grupo\n'
             '(S1–S4 = variações semânticas do mesmo tema)',
             fontsize=13, fontweight='bold', y=1.01)

plt.tight_layout()
para_path = f'{PLOTS_DIR}/05_paraphrase_heatmaps.png'
plt.savefig(para_path, bbox_inches='tight')
plt.show()
print(f'✅ Gráfico salvo em: {para_path}')

---
## Seção 5 — Teste de Textos Longos e Truncamento

O modelo `all-MiniLM-L12-v2` tem limite de **128 tokens**. Texto além desse limite é truncado.

Testamos:
- Como a **similaridade com uma query** varia conforme o documento cresce?
- Informação colocada no **início** vs. no **final** do documento é igualmente recuperada?
- Onde ocorre a queda de desempenho após o ponto de truncamento?

In [ ]:
# =============================================================
# Documento base sobre EJA — info importante no INÍCIO e no FINAL
# =============================================================

# Bloco de abertura (info importante no início — deve ser recuperada sempre)
intro = (
    "A matrícula na EJA é gratuita e aberta a qualquer pessoa com mais de 15 anos "
    "que não tenha concluído o ensino fundamental. "
)

# Bloco de enchimento neutro
filler_unit = (
    "O programa contempla diferentes modalidades pedagógicas adaptadas à realidade do estudante adulto, "
    "com carga horária flexível e conteúdo contextualizado para o mundo do trabalho e da cidadania. "
    "As escolas oferecem suporte aos alunos por meio de orientação pedagógica especializada. "
)

# Bloco de fechamento (info importante no final — será truncada em documentos longos)
outro = (
    "Ao concluir o curso, o aluno recebe certificado de conclusão do ensino médio válido em todo o Brasil."
)

# Queries
query_inicio = "Qual a idade mínima e os requisitos para matrícula gratuita na EJA?"
query_final  = "Qual documento o aluno recebe ao concluir o curso da EJA?"

# Gera versões com diferentes comprimentos
def gerar_versao(n_palavras_alvo):
    """Gera versão do documento com aproximadamente n_palavras_alvo palavras."""
    base = intro
    while len(base.split()) < n_palavras_alvo - len(outro.split()) - 5:
        base += filler_unit
    # Trunca para o alvo (ajuste fino)
    words = base.split()
    target_mid = n_palavras_alvo - len(outro.split())
    if target_mid > len(intro.split()):
        base = ' '.join(words[:target_mid])
    else:
        base = intro.rstrip()
    return base + ' ' + outro

length_targets = [20, 50, 100, 200, 300, 500]
length_versions = {n: gerar_versao(n) for n in length_targets}

print('📄 Versões geradas:')
for n, doc in length_versions.items():
    print(f'  {n:4d} palavras: {len(doc.split()):4d} palavras efetivas | trecho: "{doc[:60]}…"')

In [ ]:
# =============================================================
# Contagem de tokens por versão e cálculo de similaridades
# =============================================================
from sentence_transformers import SentenceTransformer

# Tokenizer do modelo
tokenizer = model.tokenizer

length_records = []

emb_query_inicio = encode(query_inicio)[0]
emb_query_final  = encode(query_final)[0]

for n_target, doc in length_versions.items():
    tokens = tokenizer.encode(doc)
    n_tokens = len(tokens)
    n_words  = len(doc.split())

    emb_doc = encode(doc)[0]
    sim_inicio = cos_sim(emb_doc, emb_query_inicio)
    sim_final  = cos_sim(emb_doc, emb_query_final)

    length_records.append({
        'alvo_palavras' : n_target,
        'palavras_reais': n_words,
        'tokens'        : n_tokens,
        'truncado'      : n_tokens > model.max_seq_length,
        'sim_query_inicio': round(sim_inicio, 4),
        'sim_query_final' : round(sim_final, 4),
    })

df_length = pd.DataFrame(length_records)

print('📊 Impacto do Comprimento do Texto')
print('=' * 75)
print(df_length.to_string(index=False))
print(f'\n⚠️  Limite do modelo: {model.max_seq_length} tokens')

In [ ]:
# =============================================================
# Gráfico: similaridade vs. comprimento do texto
# =============================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- Painel A: Similaridade vs. nº de tokens ---
ax = axes[0]
ax.plot(df_length['tokens'], df_length['sim_query_inicio'],
        marker='o', linewidth=2, color='#1f77b4', label='Query: Info do INÍCIO')
ax.plot(df_length['tokens'], df_length['sim_query_final'],
        marker='s', linewidth=2, color='#d62728', linestyle='--', label='Query: Info do FINAL')

# Ponto de truncamento
trunc_point = model.max_seq_length
ax.axvline(trunc_point, color='purple', linestyle=':', linewidth=2,
           label=f'Truncamento ({trunc_point} tokens)')
ax.fill_betweenx([0, 1], trunc_point, df_length['tokens'].max() + 20,
                 alpha=0.08, color='purple', label='Região truncada')

ax.set_xlabel('Número de Tokens', fontsize=11)
ax.set_ylabel('Similaridade Cosseno com a Query', fontsize=11)
ax.set_title('Similaridade vs. Comprimento (Tokens)', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
ax.set_ylim(0, 1)
ax.grid(alpha=0.3)

# Rótulos de tokens
for _, row in df_length.iterrows():
    ax.annotate(f"{int(row['tokens'])}t",
                xy=(row['tokens'], row['sim_query_inicio']),
                xytext=(4, 6), textcoords='offset points', fontsize=7.5)

# --- Painel B: Contagem de tokens vs. palavras ---
ax2 = axes[1]
ax2.bar(np.arange(len(df_length)) - 0.2,
        df_length['tokens'],   width=0.4, color='#9467bd', label='Tokens (modelo)')
ax2.bar(np.arange(len(df_length)) + 0.2,
        df_length['palavras_reais'], width=0.4, color='#8c564b', label='Palavras')
ax2.axhline(trunc_point, color='purple', linestyle=':', linewidth=2,
            label=f'Limite ({trunc_point} tokens)')
ax2.set_xticks(range(len(df_length)))
ax2.set_xticklabels([f'{n}' for n in df_length['alvo_palavras']], rotation=0)
ax2.set_xlabel('Alvo de Palavras', fontsize=11)
ax2.set_ylabel('Contagem', fontsize=11)
ax2.set_title('Tokens vs. Palavras por Versão do Documento', fontsize=12, fontweight='bold')
ax2.legend(fontsize=9)
ax2.grid(axis='y', alpha=0.3)

plt.suptitle('Impacto do Comprimento do Texto no Desempenho do Modelo\n'
             '(all-MiniLM-L12-v2 — limite de 128 tokens)',
             fontsize=13, fontweight='bold', y=1.01)

plt.tight_layout()
length_path = f'{PLOTS_DIR}/05_text_length_impact.png'
plt.savefig(length_path, bbox_inches='tight')
plt.show()
print(f'✅ Gráfico salvo em: {length_path}')

---
## Seção 6 — Teste de Mudança de Domínio

Avaliamos se o modelo mantém qualidade de recuperação fora do domínio EJA.

**Domínios testados:**
- `educacao_eja` — domínio original
- `tecnologia` — domínio técnico
- `saude` — saúde pública
- `administracao_publica` — burocracia / serviços governamentais
- `cotidiano` — linguagem informal do dia a dia

> **Métrica:** Precision@3 — proporção de documentos relevantes nos top-3 recuperados.

In [ ]:
# =============================================================
# Dados: 5 perguntas × 5 domínios + 5 documentos × 5 domínios
# =============================================================

# ---- PERGUNTAS por domínio ----
domain_questions = {
    "educacao_eja": [
        "Qual a idade mínima para se matricular na EJA?",
        "Como obter o certificado de conclusão da EJA?",
        "A EJA funciona no período noturno?",
        "Quais documentos são exigidos para a matrícula na EJA?",
        "Posso transferir minha matrícula para outra escola de EJA?"
    ],
    "tecnologia": [
        "Como funciona o algoritmo de aprendizado de máquina?",
        "Qual a diferença entre inteligência artificial e machine learning?",
        "Como treinar um modelo de linguagem natural?",
        "O que é uma rede neural convolucional?",
        "Como funciona o processamento em nuvem?"
    ],
    "saude": [
        "Como prevenir doenças cardiovasculares?",
        "Quais são os sintomas da dengue?",
        "Como funciona o sistema público de saúde no Brasil?",
        "Qual a importância da vacinação infantil?",
        "Como tratar a hipertensão arterial?"
    ],
    "administracao_publica": [
        "Como emitir a certidão de nascimento?",
        "O que é o IPTU e como é calculado?",
        "Como solicitar aposentadoria pelo INSS?",
        "Qual o prazo para pagamento do imposto de renda?",
        "Como registrar uma empresa no cartório?"
    ],
    "cotidiano": [
        "Como economizar na conta de luz em casa?",
        "Qual a melhor forma de organizar o orçamento familiar?",
        "Como fazer uma compra parcelada com segurança?",
        "O que fazer em caso de vazamento de água no apartamento?",
        "Como escolher um bom plano de celular?"
    ],
}

# ---- DOCUMENTOS por domínio (5 cada = 25 total) ----
domain_docs = {
    "educacao_eja": [
        "A EJA aceita estudantes a partir de 15 anos para o ensino fundamental e 18 anos para o ensino médio.",
        "O certificado de conclusão da EJA é emitido pela secretaria escolar após integralização da carga horária.",
        "As aulas da EJA ocorrem principalmente no período noturno, das 19h às 22h, em dias úteis.",
        "Para matrícula na EJA são necessários: RG, CPF, comprovante de residência e histórico escolar.",
        "A transferência entre escolas de EJA é permitida mediante requerimento na secretaria e apresentação de documentos."
    ],
    "tecnologia": [
        "Algoritmos de machine learning aprendem padrões a partir de dados históricos para fazer previsões.",
        "Inteligência artificial é o campo amplo; machine learning é uma subárea focada em aprendizado estatístico.",
        "Treinar um modelo de NLP envolve pré-processamento de texto, tokenização, embeddings e ajuste de hiperparâmetros.",
        "Redes neurais convolucionais são especializadas no processamento de imagens e reconhecimento de padrões visuais.",
        "Computação em nuvem permite processamento de dados em servidores remotos acessados via internet."
    ],
    "saude": [
        "Para prevenir doenças cardiovasculares recomenda-se dieta equilibrada, exercícios físicos e controle do estresse.",
        "Os sintomas da dengue incluem febre alta, dor de cabeça intensa, dores musculares e manchas na pele.",
        "O SUS (Sistema Único de Saúde) oferece atendimento gratuito a toda a população brasileira.",
        "A vacinação infantil protege contra doenças graves e contribui para a imunidade coletiva da comunidade.",
        "A hipertensão arterial é tratada com medicamentos anti-hipertensivos, mudança de hábitos e acompanhamento médico."
    ],
    "administracao_publica": [
        "A certidão de nascimento pode ser emitida gratuitamente em qualquer cartório de registro civil.",
        "O IPTU é um imposto municipal calculado sobre o valor venal do imóvel e recolhido anualmente.",
        "A aposentadoria pelo INSS pode ser solicitada pelo portal Meu INSS ou nas agências presenciais.",
        "O prazo para entrega da declaração do imposto de renda é geralmente até o último dia de abril.",
        "O registro de empresas é feito na Junta Comercial do estado onde a empresa será estabelecida."
    ],
    "cotidiano": [
        "Para economizar energia elétrica, desligue aparelhos em stand-by e prefira lâmpadas de LED.",
        "Uma planilha de orçamento familiar ajuda a controlar receitas e despesas mensais com clareza.",
        "Ao comprar parcelado, verifique a taxa de juros e compare o preço final com o valor à vista.",
        "Em caso de vazamento de água, feche o registro geral e acione o serviço de manutenção do condomínio.",
        "Para escolher um plano de celular, considere cobertura da operadora, franquia de dados e custo mensal."
    ],
}

# Monta corpus misto com rótulo de domínio
all_docs    = []
all_labels  = []  # domínio de cada documento
for dom, docs in domain_docs.items():
    for doc in docs:
        all_docs.append(doc)
        all_labels.append(dom)

print(f'✅ Corpus misto: {len(all_docs)} documentos em {len(domain_docs)} domínios')

In [ ]:
# =============================================================
# Codificação do corpus misto e cálculo de Precision@3
# =============================================================
corpus_mixed_embs = encode(all_docs)
all_labels_arr    = np.array(all_labels)

TOP_K = 3
domain_results = []

for domain, questions in domain_questions.items():
    q_embs = encode(questions)
    sims   = cosine_similarity(q_embs, corpus_mixed_embs)  # (5, 25)

    precisions = []
    for q_idx in range(len(questions)):
        top_k_idx = np.argsort(sims[q_idx])[::-1][:TOP_K]
        retrieved_domains = all_labels_arr[top_k_idx]
        # Precision@K = proporção de docs relevantes (mesmo domínio)
        p_at_k = (retrieved_domains == domain).sum() / TOP_K
        precisions.append(p_at_k)

    mean_p = np.mean(precisions)
    domain_results.append({
        'dominio'      : domain,
        'precision_at3': round(float(mean_p), 4),
        'precisions_q' : [round(p, 2) for p in precisions]
    })

df_domain = pd.DataFrame(domain_results)

print('📊 Precision@3 por Domínio')
print('=' * 55)
for _, row in df_domain.iterrows():
    bar = '█' * int(row['precision_at3'] * 20)
    print(f'  {row["dominio"]:25s}: {row["precision_at3"]:.4f}  {bar}')
    print(f'    Por questão: {row["precisions_q"]}')
print(f'\n📈 Precision@3 média global: {df_domain["precision_at3"].mean():.4f}')

In [ ]:
# =============================================================
# Gráfico: Precision@3 por domínio
# =============================================================
domain_labels_pt = {
    'educacao_eja'         : 'Educação\nEJA',
    'tecnologia'           : 'Tecnologia',
    'saude'                : 'Saúde',
    'administracao_publica': 'Adm.\nPública',
    'cotidiano'            : 'Cotidiano'
}

df_domain['label_pt'] = df_domain['dominio'].map(domain_labels_pt)

palette = ['#2ca02c' if p >= 0.80 else ('#ff7f0e' if p >= 0.60 else '#d62728')
           for p in df_domain['precision_at3']]

fig, ax = plt.subplots(figsize=(10, 6))

bars = ax.bar(df_domain['label_pt'], df_domain['precision_at3'],
              color=palette, edgecolor='white', width=0.55)

# Rótulos de valor
for bar, val in zip(bars, df_domain['precision_at3']):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.015,
            f'{val:.2f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

# Referências
ax.axhline(1.0, color='green',   linestyle='--', linewidth=1.2, alpha=0.6, label='Perfeito (1.0)')
ax.axhline(0.70, color='orange', linestyle=':', linewidth=1.5,             label='Aceitável (0.70)')

ax.set_ylim(0, 1.15)
ax.set_xlabel('Domínio', fontsize=12)
ax.set_ylabel('Precision@3', fontsize=12)
ax.set_title('Precision@3 por Domínio — Teste de Mudança de Domínio\n'
             '(corpus misto com 25 documentos: 5 por domínio)',
             fontsize=12, fontweight='bold', pad=12)
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)

# Legenda de cores
patches_dom = [
    mpatches.Patch(color='#2ca02c', label='Bom (≥ 0.80)'),
    mpatches.Patch(color='#ff7f0e', label='Médio (≥ 0.60)'),
    mpatches.Patch(color='#d62728', label='Ruim (< 0.60)'),
]
legend_colors = ax.legend(handles=patches_dom, loc='upper right',
                           fontsize=9, title='Desempenho', title_fontsize=9)
ax.add_artist(legend_colors)
ax.legend(fontsize=10, loc='upper left')

plt.tight_layout()
domain_path = f'{PLOTS_DIR}/05_domain_shift.png'
plt.savefig(domain_path, bbox_inches='tight')
plt.show()
print(f'✅ Gráfico salvo em: {domain_path}')

---
## Seção 7 — Sumário de Robustez

Consolidação de todas as métricas avaliadas, com **nível de risco** para implantação em RAG.

In [ ]:
# =============================================================
# Construção da tabela de resumo de robustez
# =============================================================

# --- Métricas ortográficas ---
mean_sem_acento  = df_ortho[df_ortho['variante_tipo']=='sem_acento']['similaridade'].mean()
mean_minuscula   = df_ortho[df_ortho['variante_tipo']=='minuscula']['similaridade'].mean()
mean_erro_tipico = df_ortho[df_ortho['variante_tipo']=='erro_tipico']['similaridade'].mean()

# --- Métricas de negação ---
mean_neg    = df_neg['similaridade'].mean()
n_danger    = (df_neg['similaridade'] > 0.85).sum()

# --- Métricas de paráfrases ---
mean_para   = df_para['mean_intra_sim'].mean()
mean_consist= df_para['retrieval_consistency'].mean()

# --- Métricas de comprimento ---
sim_inicio_longo = df_length[df_length['tokens'] > 128]['sim_query_inicio'].mean() \
    if (df_length['tokens'] > 128).any() else df_length['sim_query_inicio'].iloc[-1]
sim_final_longo  = df_length[df_length['tokens'] > 128]['sim_query_final'].mean() \
    if (df_length['tokens'] > 128).any() else df_length['sim_query_final'].iloc[-1]

# --- Métricas de domínio ---
p3_eja    = df_domain[df_domain['dominio']=='educacao_eja']['precision_at3'].values[0]
p3_outros = df_domain[df_domain['dominio']!='educacao_eja']['precision_at3'].mean()

summary = [
    {
        "teste"        : "Ortografia — Sem Acento",
        "n_exemplos"   : 10,
        "metrica_chave": "sim_media_vs_original",
        "valor"        : round(float(mean_sem_acento), 4),
        "risco"        : "Baixo" if mean_sem_acento >= 0.90 else ("Médio" if mean_sem_acento >= 0.80 else "Alto"),
        "notas"        : "Remoção de acentos tem impacto mínimo no modelo."
    },
    {
        "teste"        : "Ortografia — Minúscula S/ Acento",
        "n_exemplos"   : 10,
        "metrica_chave": "sim_media_vs_original",
        "valor"        : round(float(mean_minuscula), 4),
        "risco"        : "Baixo" if mean_minuscula >= 0.90 else ("Médio" if mean_minuscula >= 0.80 else "Alto"),
        "notas"        : "Caixa baixa + sem acentos ainda é bem compreendido pelo modelo."
    },
    {
        "teste"        : "Ortografia — Erros Típicos",
        "n_exemplos"   : 10,
        "metrica_chave": "sim_media_vs_original",
        "valor"        : round(float(mean_erro_tipico), 4),
        "risco"        : "Baixo" if mean_erro_tipico >= 0.90 else ("Médio" if mean_erro_tipico >= 0.80 else "Alto"),
        "notas"        : "Erros de digitação comuns afetam mais do que simples remoção de acentos."
    },
    {
        "teste"        : "Negação — Similaridade Afirmação/Negação",
        "n_exemplos"   : 20,
        "metrica_chave": "sim_media_pares_negados",
        "valor"        : round(float(mean_neg), 4),
        "risco"        : "Alto",
        "notas"        : f"{n_danger}/20 pares com sim>0.85. RISCO CRÍTICO: modelo não distingue bem negações em RAG."
    },
    {
        "teste"        : "Paráfrases — Coesão Intra-Grupo",
        "n_exemplos"   : 8,
        "metrica_chave": "sim_media_intra_grupo",
        "valor"        : round(float(mean_para), 4),
        "risco"        : "Baixo" if mean_para >= 0.80 else "Médio",
        "notas"        : "Paráfrases semânticas do mesmo tema são bem agrupadas."
    },
    {
        "teste"        : "Paráfrases — Consistência de Recuperação",
        "n_exemplos"   : 8,
        "metrica_chave": "consistencia_top3",
        "valor"        : round(float(mean_consist), 4),
        "risco"        : "Baixo" if mean_consist >= 0.80 else "Médio",
        "notas"        : "Variantes de uma mesma pergunta recuperam documentos similares."
    },
    {
        "teste"        : "Texto Longo — Query no Início (pós-truncamento)",
        "n_exemplos"   : 6,
        "metrica_chave": "sim_query_inicio_longo",
        "valor"        : round(float(sim_inicio_longo), 4),
        "risco"        : "Baixo" if sim_inicio_longo >= 0.60 else "Médio",
        "notas"        : "Info no início do doc mantém similaridade após truncamento."
    },
    {
        "teste"        : "Texto Longo — Query no Final (pós-truncamento)",
        "n_exemplos"   : 6,
        "metrica_chave": "sim_query_final_longo",
        "valor"        : round(float(sim_final_longo), 4),
        "risco"        : "Alto" if sim_final_longo < 0.40 else ("Médio" if sim_final_longo < 0.60 else "Baixo"),
        "notas"        : "Info no FINAL do doc é perdida após truncamento. Risco para RAG com docs longos."
    },
    {
        "teste"        : "Mudança de Domínio — EJA (domínio original)",
        "n_exemplos"   : 5,
        "metrica_chave": "precision_at3",
        "valor"        : round(float(p3_eja), 4),
        "risco"        : "Baixo" if p3_eja >= 0.80 else "Médio",
        "notas"        : "Desempenho no domínio original de treinamento/uso."
    },
    {
        "teste"        : "Mudança de Domínio — Outros Domínios (média)",
        "n_exemplos"   : 20,
        "metrica_chave": "precision_at3",
        "valor"        : round(float(p3_outros), 4),
        "risco"        : "Baixo" if p3_outros >= 0.80 else ("Médio" if p3_outros >= 0.60 else "Alto"),
        "notas"        : "Degradação de desempenho fora do domínio EJA. Verificar adequação para outros contextos."
    },
]

df_summary = pd.DataFrame(summary)

print('📋 SUMÁRIO FINAL DE ROBUSTEZ')
print('=' * 90)
print(df_summary[['teste','n_exemplos','metrica_chave','valor','risco']].to_string(index=False))

In [ ]:
# =============================================================
# Salvar resultados em JSON
# =============================================================
results_json = {
    "modelo"          : MODEL_NAME,
    "notebook"        : "05_robustness.ipynb",
    "max_seq_length"  : model.max_seq_length,
    "embedding_dim"   : model.get_sentence_embedding_dimension(),
    "testes"          : summary,
    "detalhes": {
        "ortografia": df_ortho.to_dict(orient='records'),
        "negacao"   : df_neg.drop(columns=['risco']).to_dict(orient='records'),
        "parafrases": df_para.to_dict(orient='records'),
        "comprimento": df_length.to_dict(orient='records'),
        "dominio"   : df_domain.drop(columns=['label_pt'], errors='ignore').to_dict(orient='records')
    }
}

json_path = f'{METRICS_DIR}/05_robustness_results.json'
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(results_json, f, ensure_ascii=False, indent=2)

print(f'✅ Resultados salvos em: {json_path}')

In [ ]:
# =============================================================
# Gráfico: painel geral do sumário de robustez
# =============================================================
risco_color = {'Baixo': '#2ca02c', 'Médio': '#ff7f0e', 'Alto': '#d62728'}
cores_sumario = df_summary['risco'].map(risco_color).tolist()

fig, ax = plt.subplots(figsize=(14, 7))

y_pos = np.arange(len(df_summary))
bars  = ax.barh(y_pos, df_summary['valor'], color=cores_sumario,
                edgecolor='white', height=0.6)

# Rótulos de valor
for bar, val, risco in zip(bars, df_summary['valor'], df_summary['risco']):
    ax.text(max(val + 0.01, 0.02), bar.get_y() + bar.get_height() / 2,
            f'{val:.4f}', va='center', fontsize=8.5, fontweight='bold')

ax.set_yticks(y_pos)
ax.set_yticklabels(
    [f"{row['teste']} (n={row['n_exemplos']})" for _, row in df_summary.iterrows()],
    fontsize=8.5
)
ax.set_xlim(0, 1.15)
ax.set_xlabel('Valor da Métrica', fontsize=11)
ax.set_title('Sumário de Robustez — all-MiniLM-L12-v2\n'
             'Visão geral de todos os testes realizados',
             fontsize=13, fontweight='bold', pad=12)

patches_risk = [
    mpatches.Patch(color='#2ca02c', label='Risco Baixo'),
    mpatches.Patch(color='#ff7f0e', label='Risco Médio'),
    mpatches.Patch(color='#d62728', label='Risco Alto'),
]
ax.legend(handles=patches_risk, loc='lower right', fontsize=10)
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/05_robustness_summary.png', bbox_inches='tight')
plt.show()
print('✅ Gráfico de sumário salvo.')

---
## 📝 Interpretação e Recomendações

### ✅ Pontos Fortes do Modelo

| Aspecto | Avaliação |
|---------|----------|
| **Robustez ortográfica** | Alta — acentos e caixa têm impacto mínimo |
| **Paráfrases** | Alta — sentenças semanticamente equivalentes são corretamente agrupadas |
| **Domínio EJA** | Alta Precision@3 no domínio de treinamento/uso |

---

### ⚠️ Limitações e Riscos para RAG

#### 🔴 Negação — Risco Crítico

> Em modelos de embedding como o `all-MiniLM-L12-v2`, a negação **raramente altera**  
> de forma significativa o vetor de embedding. Isso ocorre porque o modelo foi treinado  
> para maximizar similaridade semântica geral, não para capturar polaridade lógica.

**Implicação prática para RAG:**  
Se um usuário pergunta *"O aluno **não** pode se matricular na EJA?"*, o sistema pode  
recuperar o documento *"O aluno **pode** se matricular na EJA"* com alta similaridade  
— fornecendo ao LLM informação com sentido **oposto** ao buscado.

**Mitigações recomendadas:**
- 🔧 Adicionar **filtros de negação** na camada de recuperação (detectar "não", "nunca", "jamais")
- 🔧 Usar **re-rankers** (cross-encoders) que entendem polaridade semântica
- 🔧 Incluir **exemplos negativos** explícitos no corpus para diferenciar casos
- 🔧 Implementar verificação de consistência lógica no pós-processamento do LLM

---

#### 🟠 Truncamento de Texto Longo

Documentos com mais de **128 tokens** sofrem truncamento silencioso.  
Informações posicionadas no **final do documento** são completamente descartadas.  

**Mitigações:**
- 🔧 Chunking de documentos longos em blocos de ≤ 100 tokens com sobreposição
- 🔧 Colocar informações-chave sempre no **início** de cada chunk
- 🔧 Usar modelos com janela maior (ex.: `all-mpnet-base-v2`, `e5-large`) para docs extensos

---

#### 🟡 Degradação por Mudança de Domínio

O desempenho pode cair em domínios muito diferentes do EJA.  
Para aplicações multi-domínio, avaliar fine-tuning no domínio alvo.

---

### 📊 Resumo de Risco Geral

```
🟢 Ortografia / Acentuação  → BAIXO RISCO  
🔴 Negação                  → ALTO RISCO CRÍTICO  
🟢 Paráfrases               → BAIXO RISCO  
🟠 Textos Longos            → MÉDIO / ALTO (info no final perde-se)  
🟡 Mudança de Domínio       → MÉDIO (monitorar)  
```